# Kernel Derivation for the Sigmoid Envelope (Eq. 11)

This notebook derives the GP kernel components for the **sigmoid** spot emergence/decay profile
(Eq. 11, `alpha_sigmoid`):

$$\alpha_{\rm smooth}(t) = \frac{\alpha_{\max}}{C}\;
\frac{1}{\big(1 + e^{-t/\tau_{\rm em}}\big)\,\big(1 + e^{(t - \tau_{\rm plat})/\tau_{\rm dec}}\big)}$$

where $C = \max_t\!\left[\big(1 + e^{-t/\tau_{\rm em}}\big)^{-1}\big(1 + e^{(t-\tau_{\rm plat})/\tau_{\rm dec}}\big)^{-1}\right]$ normalizes the peak to $\alpha_{\max}$.

Unlike the trapezoidal envelope, the sigmoid is $C^\infty$ everywhere, giving a smooth $R_\Gamma(\tau)$
with no derivative discontinuities.

**Steps:**
1. Define the sigmoid envelope and its squared version $\Gamma(t) = \alpha_{\rm smooth}^2(t)$
2. Compute $\hat{\Gamma}(\omega)$ via numerical FFT (no closed-form exists)
3. Compute $R_\Gamma(\tau) = \mathcal{F}^{-1}\{|\hat{\Gamma}(\omega)|^2\}$ via the Wiener–Khinchin theorem
4. Assemble the full single-spot kernel $k(\tau) = \alpha_{\max}^4 R_\Gamma(\tau)\left[|c_0|^2 + 2\sum_{n\geq1}|c_n|^2\cos(n\omega_0\tau)\right]$
5. Compare to the trapezoidal kernel

**Key result:** The sigmoid kernel has the same factorized structure as the trapezoidal kernel
(Eq. `kernel_single_spot`), but with a smoother $R_\Gamma(\tau)$ that must be evaluated numerically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.signal import fftconvolve

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12

## 1. The Sigmoid Envelope

The sigmoid envelope is a product of two logistic functions:
$$\alpha_{\rm smooth}(t) = \frac{\alpha_{\max}}{C} \cdot \sigma\!\left(\frac{t}{\tau_{\rm em}}\right) \cdot \sigma\!\left(\frac{\tau_{\rm plat} - t}{\tau_{\rm dec}}\right)$$
where $\sigma(x) = 1/(1+e^{-x})$ is the standard logistic function.

This is $C^\infty$ everywhere and has the same asymptotic behavior as the
piecewise exponential model (Eq. 10):
- $t \ll 0$: $\alpha \approx \alpha_{\max} e^{t/\tau_{\rm em}}$
- $0 \ll t \ll \tau_{\rm plat}$: $\alpha \approx \alpha_{\max}$
- $t \gg \tau_{\rm plat}$: $\alpha \approx \alpha_{\max} e^{-(t-\tau_{\rm plat})/\tau_{\rm dec}}$

In [ ]:
def sigmoid(x):
    """Standard logistic sigmoid."""
    return 1.0 / (1.0 + np.exp(-x))


def alpha_sigmoid(t, tau_em, tau_plat, tau_dec, alpha_max=1.0):
    """Sigmoid envelope (Eq. 11).
    
    Parameters
    ----------
    t : array
        Time array.
    tau_em : float
        Emergence e-folding timescale.
    tau_plat : float
        Plateau duration.
    tau_dec : float
        Decay e-folding timescale.
    alpha_max : float
        Peak angular radius.
    
    Returns
    -------
    alpha : array
        Spot angular radius at each time.
    """
    rise = sigmoid(t / tau_em)
    fall = sigmoid((tau_plat - t) / tau_dec)
    raw = rise * fall
    C = np.max(raw)
    return alpha_max * raw / C


def normalization_constant(tau_em, tau_plat, tau_dec):
    """Compute the normalization constant C analytically.
    
    The peak of sigma(t/tau_em) * sigma((tau_plat-t)/tau_dec) occurs at
    t* = tau_plat * tau_em / (tau_em + tau_dec) when tau_em = tau_dec,
    and in general at the solution of d/dt[...] = 0.
    """
    res = minimize_scalar(
        lambda t: -sigmoid(t / tau_em) * sigmoid((tau_plat - t) / tau_dec),
        bounds=(-5 * tau_em, tau_plat + 5 * tau_dec),
        method='bounded'
    )
    return -res.fun

In [ ]:
# Example parameters
tau_em = 3.0     # emergence timescale [days]
tau_plat = 15.0   # plateau duration [days]
tau_dec = 3.0    # decay timescale [days]
alpha_max = 0.1  # peak angular radius [rad]

# Time grid (wide enough to capture the full envelope)
t_margin = 8 * max(tau_em, tau_dec)
t = np.linspace(-t_margin, tau_plat + t_margin, 5000)

alpha = alpha_sigmoid(t, tau_em, tau_plat, tau_dec, alpha_max)

fig, ax = plt.subplots(1, 1, figsize=(10, 3.5))
ax.plot(t, alpha, 'C0', lw=2, label=r'$\alpha_{\rm smooth}(t)$')
ax.axhline(alpha_max, color='gray', ls='--', lw=0.8, label=r'$\alpha_{\max}$')
ax.axvline(0, color='gray', ls=':', lw=0.8)
ax.axvline(tau_plat, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Time [days]')
ax.set_ylabel(r'$\alpha(t)$ [rad]')
ax.set_title('Sigmoid Envelope (Eq. 11)')
ax.legend()
ax.set_xlim(t[0], t[-1])
plt.tight_layout()
plt.show()

### Check: normalization $C$ and peak location

`alpha_sigmoid` takes $C$ as the maximum of the unnormalized product on the time grid. It must agree with the
optimizer-based `normalization_constant` to grid resolution (the grid spacing is 0.013 d and the error is
quadratic in it near the flat peak), the peak of $\alpha$ must equal $\alpha_{\max}$, and for
$\tau_{\rm em} = \tau_{\rm dec}$ the peak lies at $t^* = \tau_{\rm plat}/2$ (docstring of `normalization_constant`).

In [ ]:
raw_grid = sigmoid(t / tau_em) * sigmoid((tau_plat - t) / tau_dec)
C_grid = np.max(raw_grid)
C_opt = normalization_constant(tau_em, tau_plat, tau_dec)
print(f"C (grid max) = {C_grid:.10f}, C (optimizer) = {C_opt:.10f}, relative difference = {C_grid / C_opt - 1:.2e}")

assert np.isclose(np.max(alpha), alpha_max, rtol=1e-12, atol=0), "peak of alpha is not alpha_max"
assert abs(C_grid / C_opt - 1) < 1e-5, "grid normalization disagrees with normalization_constant"

# symmetric example: the maximum sits at t* = tau_plat / 2
assert tau_em == tau_dec
C_mid = sigmoid(tau_plat / 2 / tau_em) * sigmoid(tau_plat / 2 / tau_dec)
assert abs(C_mid / C_opt - 1) < 1e-9, "peak of the symmetric envelope is not at tau_plat/2"
print("Verified: peak = alpha_max, C consistent, symmetric peak at tau_plat/2.")

## 2. The Squared Envelope $\Gamma(t) = \alpha^2_{\rm smooth}(t)$

The kernel requires the **squared** envelope:
$$\Gamma(t) = \alpha_{\rm smooth}^2(t) = \frac{\alpha_{\max}^2}{C^2} \cdot \sigma^2\!\left(\frac{t}{\tau_{\rm em}}\right) \cdot \sigma^2\!\left(\frac{\tau_{\rm plat} - t}{\tau_{\rm dec}}\right)$$

Since $\sigma^2(x) = \sigma(x) - \sigma(x)[1-\sigma(x)] = \sigma(x)(2\sigma(x)-1)/1$, squaring
preserves the $C^\infty$ smoothness. The squared envelope has the same support and
asymptotic behavior, but with doubled exponential rates in the tails:
- $t \ll 0$: $\Gamma(t) \approx \alpha_{\max}^2 e^{2t/\tau_{\rm em}}$
- $t \gg \tau_{\rm plat}$: $\Gamma(t) \approx \alpha_{\max}^2 e^{-2(t-\tau_{\rm plat})/\tau_{\rm dec}}$

In [ ]:
def Gamma_sigmoid(t, tau_em, tau_plat, tau_dec, alpha_max=1.0):
    """Squared sigmoid envelope: Gamma(t) = alpha_smooth(t)^2."""
    alpha = alpha_sigmoid(t, tau_em, tau_plat, tau_dec, alpha_max)
    return alpha**2


Gamma = Gamma_sigmoid(t, tau_em, tau_plat, tau_dec, alpha_max)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].plot(t, alpha, 'C0', lw=2, label=r'$\alpha(t)$')
axes[0].plot(t, np.sqrt(Gamma), 'C1', lw=2, ls='--', label=r'$\sqrt{\Gamma(t)}$')
axes[0].set_ylabel(r'$\alpha(t)$')
axes[0].legend()
axes[0].set_title(r'$\alpha(t)$ vs $\sqrt{\Gamma(t)}$ (should overlap)')

axes[1].plot(t, Gamma, 'C1', lw=2, label=r'$\Gamma(t) = \alpha^2(t)$')
axes[1].set_ylabel(r'$\Gamma(t)$')
axes[1].legend()
axes[1].set_title(r'Squared Envelope $\Gamma(t)$')

for ax in axes:
    ax.set_xlabel('Time [days]')
    ax.set_xlim(t[0], t[-1])

plt.tight_layout()
plt.show()

### Check: $\sqrt{\Gamma} = \alpha$ and the tail e-folding rates

The two curves in the left panel must overlap, and the asymptotic forms claimed in Sections 1–2 require
$d\ln\Gamma/dt \to 2/\tau_{\rm em}$ for $t \ll 0$ and $\to -2/\tau_{\rm dec}$ for $t \gg \tau_{\rm plat}$. At the edges
of a grid spanning $8\max(\tau_{\rm em}, \tau_{\rm dec})$ beyond the plateau, the logistic correction to the slope
is $\approx e^{-8} \approx 3.4\times10^{-4}$, so we use a $10^{-3}$ relative tolerance. Tested for the example
parameters and an asymmetric case $(\tau_{\rm em}, \tau_{\rm dec}) = (1, 5)$ d.

In [ ]:
assert np.allclose(np.sqrt(Gamma), alpha, rtol=1e-12, atol=0), "sqrt(Gamma) != alpha"

for te_c, td_c in [(tau_em, tau_dec), (1.0, 5.0)]:
    margin_c = 8 * max(te_c, td_c)
    t_c = np.linspace(-margin_c, tau_plat + margin_c, 5000)
    dlnG = np.gradient(np.log(Gamma_sigmoid(t_c, te_c, tau_plat, td_c, alpha_max)), t_c)
    rise_dev = dlnG[0] * te_c / 2 - 1
    decay_dev = -dlnG[-1] * td_c / 2 - 1
    print(f"tau_em={te_c}, tau_dec={td_c}: tail slope deviations {rise_dev:.2e} (rise), {decay_dev:.2e} (decay)")
    assert abs(rise_dev) < 1e-3, "rise tail of Gamma does not e-fold at 2/tau_em"
    assert abs(decay_dev) < 1e-3, "decay tail of Gamma does not e-fold at 2/tau_dec"

## 3. Fourier Transform $\hat{\Gamma}(\omega)$

Unlike the trapezoidal envelope (Eq. `Gamma_hat`), the sigmoid envelope does **not** have a
closed-form Fourier transform (noted as "(numerical)" in Table 2 of the paper).

We compute $\hat{\Gamma}(\omega)$ numerically via FFT. Since $\Gamma(t)$ is real and
effectively even about its center $t = \tau_{\rm plat}/2$, the transform is well-behaved.

In [ ]:
def compute_Gamma_hat_fft(t_arr, Gamma_arr):
    """Compute the Fourier transform of Gamma(t) via FFT.
    
    Returns angular frequencies omega and the (complex) FT values.
    We use the convention: Gamma_hat(omega) = int Gamma(t) exp(-i*omega*t) dt.
    """
    dt = t_arr[1] - t_arr[0]
    N = len(t_arr)
    
    # FFT (numpy convention: sum_n x_n exp(-2*pi*i*n*k/N))
    Gamma_fft = np.fft.fft(Gamma_arr) * dt
    freq = np.fft.fftfreq(N, d=dt)  # cycles per unit time
    omega = 2 * np.pi * freq         # angular frequency
    
    # Sort by frequency
    idx = np.argsort(omega)
    return omega[idx], Gamma_fft[idx]


def compute_Gamma_hat_direct(omega_arr, t_arr, Gamma_arr):
    """Compute Gamma_hat(omega) by direct numerical integration (for validation)."""
    dt = t_arr[1] - t_arr[0]
    result = np.empty(len(omega_arr), dtype=complex)
    for i, w in enumerate(omega_arr):
        integrand = Gamma_arr * np.exp(-1j * w * t_arr)
        result[i] = np.trapezoid(integrand, t_arr)
    return result

In [ ]:
# Use a fine, wide grid for accurate FFT
t_fine = np.linspace(-t_margin, tau_plat + t_margin, 2**14)
Gamma_fine = Gamma_sigmoid(t_fine, tau_em, tau_plat, tau_dec, alpha_max)

omega_fft, Gamma_hat_fft = compute_Gamma_hat_fft(t_fine, Gamma_fine)

# Validate against direct integration at a few frequencies
omega_test = np.linspace(0.01, 5.0, 50)
Gamma_hat_direct = compute_Gamma_hat_direct(omega_test, t_fine, Gamma_fine)

# Since Gamma is real, Gamma_hat is conjugate-symmetric.
# For a real, approximately even function, the imaginary part should be small.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

mask = (omega_fft > 0) & (omega_fft < 6)
axes[0].plot(omega_fft[mask], np.abs(Gamma_hat_fft[mask])**2, 'C0', lw=1.5,
             label='FFT')
axes[0].plot(omega_test, np.abs(Gamma_hat_direct)**2, 'C1o', ms=4, alpha=0.7,
             label='Direct integration')
axes[0].set_xlabel(r'$\omega$ [rad/day]')
axes[0].set_ylabel(r'$|\hat{\Gamma}(\omega)|^2$')
axes[0].set_title(r'Energy Spectral Density $|\hat{\Gamma}(\omega)|^2$')
axes[0].set_yscale('log')
axes[0].legend()

# Relative error
from scipy.interpolate import interp1d
Gamma_hat_interp = interp1d(omega_fft, Gamma_hat_fft, kind='linear')
Gamma_hat_fft_at_test = Gamma_hat_interp(omega_test)
rel_err = np.abs(Gamma_hat_fft_at_test - Gamma_hat_direct) / np.abs(Gamma_hat_direct)

axes[1].semilogy(omega_test, rel_err, 'C2.-')
axes[1].set_xlabel(r'$\omega$ [rad/day]')
axes[1].set_ylabel('Relative error')
axes[1].set_title('FFT vs Direct Integration')
axes[1].axhline(1e-3, color='gray', ls='--', lw=0.8)

plt.tight_layout()
plt.show()

print(f"Max relative error (FFT vs direct): {np.max(rel_err):.2e}")

### Check: FFT vs direct integration

The relative error printed above is not a meaningful accuracy measure. `np.fft.fft` places $t = 0$ at the first
grid point, so `compute_Gamma_hat_fft` returns $e^{i\omega t_0}\hat{\Gamma}(\omega)$ with $t_0 = $ `t_fine[0]`
$= -24$ d rather than the $\hat{\Gamma}(\omega)$ of its docstring, and the oscillating complex spectrum is then
linearly interpolated between FFT bins. The magnitude, which is all that enters $|\hat{\Gamma}|^2$ and $R_\Gamma$, is
unaffected. Here we compare at the FFT bins themselves ($0 \le \omega \le 5$), (i) with the phase factor
$e^{-i\omega t_0}$ restored and (ii) in magnitude. FFT (Riemann sum) and trapezoid rule differ only by the endpoint
terms, where $\Gamma \sim 10^{-7}\alpha_{\max}^2$, so the agreement should be far below $10^{-8}\hat{\Gamma}(0)$;
a wrong frequency mapping, scaling, or sign convention gives an $O(1)$ error. (iii) For this symmetric example
$\Gamma$ is even about $\tau_{\rm plat}/2$, so $\hat{\Gamma}(\omega)\,e^{i\omega\tau_{\rm plat}/2}$ must be real.

In [ ]:
t0_fine = t_fine[0]
Gamma_hat_0 = np.trapezoid(Gamma_fine, t_fine)        # Gamma_hat(0) = area under Gamma

on_bins = (omega_fft >= 0) & (omega_fft <= 5.0)
w_bins = omega_fft[on_bins]
Gh_direct_bins = compute_Gamma_hat_direct(w_bins, t_fine, Gamma_fine)
Gh_fft_bins = Gamma_hat_fft[on_bins]

err_phased = np.max(np.abs(Gh_fft_bins * np.exp(-1j * w_bins * t0_fine) - Gh_direct_bins)) / Gamma_hat_0
err_mag = np.max(np.abs(np.abs(Gh_fft_bins) - np.abs(Gh_direct_bins))) / Gamma_hat_0
err_even = np.max(np.abs(np.imag(Gh_direct_bins * np.exp(1j * w_bins * tau_plat / 2)))) / Gamma_hat_0
print(f"{on_bins.sum()} bins: phased error {err_phased:.2e}, magnitude error {err_mag:.2e}, "
      f"Im[centred transform] {err_even:.2e} (all relative to Gamma_hat(0))")

assert err_phased < 1e-8, "FFT does not reproduce the direct Fourier integral"
assert err_mag < 1e-8, "|FFT| does not reproduce |Gamma_hat|"
assert tau_em == tau_dec
assert err_even < 1e-8, "symmetric envelope: Gamma_hat * exp(i w tau_plat/2) is not real"

## 4. Autocorrelation $R_\Gamma(\tau)$ via the Wiener–Khinchin Theorem

By the Wiener–Khinchin theorem (Theorem 3 in the paper):
$$R_\Gamma(\tau) = \frac{1}{2\pi}\int_{-\infty}^{\infty} |\hat{\Gamma}(\omega)|^2 e^{i\omega\tau}\,d\omega$$

Equivalently, $R_\Gamma(\tau)$ can be computed as the direct convolution:
$$R_\Gamma(\tau) = \int_{-\infty}^{\infty} \Gamma(s)\,\Gamma(s+\tau)\,ds$$

We compute both and verify they agree.

In [ ]:
def R_Gamma_fft(t_arr, Gamma_arr):
    """Autocorrelation via Wiener-Khinchin: R = IFFT(|FFT(Gamma)|^2)."""
    dt = t_arr[1] - t_arr[0]
    N = len(t_arr)
    Gamma_fft = np.fft.fft(Gamma_arr)
    esd = np.abs(Gamma_fft)**2
    R = np.fft.ifft(esd).real * dt
    # Build the lag array
    tau_arr = np.arange(N) * dt
    return tau_arr, R


def R_Gamma_convolution(t_arr, Gamma_arr, tau_arr):
    """Autocorrelation via direct convolution: R(tau) = int Gamma(s) Gamma(s+tau) ds."""
    dt = t_arr[1] - t_arr[0]
    R = np.empty_like(tau_arr)
    for i, tau_val in enumerate(tau_arr):
        # Shift the given Gamma by tau (zero outside the grid)
        Gamma_shifted = np.interp(t_arr + tau_val, t_arr, Gamma_arr, left=0.0, right=0.0)
        R[i] = np.trapezoid(Gamma_arr * Gamma_shifted, t_arr)
    return R

In [ ]:
# Compute R_Gamma via FFT
tau_fft, R_fft = R_Gamma_fft(t_fine, Gamma_fine)

# Compute R_Gamma via direct convolution (at selected points for validation)
tau_test = np.linspace(0, tau_plat + 4 * max(tau_em, tau_dec), 200)
R_conv = R_Gamma_convolution(t_fine, Gamma_fine, tau_test)

# Normalize: R_Gamma_normalized(tau) = R_Gamma(tau) / R_Gamma(0)
R_fft_norm = R_fft / R_fft[0]
R_conv_norm = R_conv / R_conv[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

tau_plot_max = tau_plat + 5 * max(tau_em, tau_dec)
mask = tau_fft < tau_plot_max

axes[0].plot(tau_fft[mask], R_fft_norm[mask], 'C0', lw=2, label='FFT (Wiener–Khinchin)')
axes[0].plot(tau_test, R_conv_norm, 'C1o', ms=3, alpha=0.6, label='Direct convolution')
axes[0].set_xlabel(r'$\tau$ [days]')
axes[0].set_ylabel(r'$R_\Gamma(\tau) / R_\Gamma(0)$')
axes[0].set_title(r'Normalized Envelope Autocorrelation $R_\Gamma(\tau)$')
axes[0].legend()
axes[0].set_xlim(0, tau_plot_max)

# Residual
from scipy.interpolate import interp1d
R_fft_interp = interp1d(tau_fft, R_fft_norm, kind='linear')
R_fft_at_test = R_fft_interp(tau_test)
axes[1].plot(tau_test, R_fft_at_test - R_conv_norm, 'C2', lw=1.5)
axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel('FFT – Convolution')
axes[1].set_title('Residual (FFT vs Convolution)')
axes[1].axhline(0, color='gray', ls='--', lw=0.8)
axes[1].set_xlim(0, tau_plot_max)

plt.tight_layout()
plt.show()

max_resid = np.max(np.abs(R_fft_at_test - R_conv_norm))
print(f"Max |residual| between FFT and convolution methods: {max_resid:.2e}")
print(f"R_Gamma(0) = {R_fft[0]:.6e}")

# The FFT gives the circular autocorrelation over the 63 d window; its wrap-around grows with
# lag and reaches ~2.5e-6 R(0) at the largest test lag (27 d), so 1e-5 leaves a 4x margin.
assert max_resid < 1e-5, f"Wiener-Khinchin and direct convolution disagree: {max_resid:.2e}"
# Unnormalized: both must give R(0) = int Gamma^2 dt
assert np.isclose(R_fft[0], R_conv[0], rtol=1e-9, atol=0), (R_fft[0], R_conv[0])

## 5. Smoothness: Sigmoid vs Trapezoidal $R_\Gamma(\tau)$

A key advantage of the sigmoid envelope is that $R_\Gamma(\tau)$ is $C^\infty$,
whereas the trapezoidal envelope produces a piecewise polynomial $R_\Gamma(\tau)$ with
kinks (derivative discontinuities) at $\tau = \tau_s$, $\ell$, $\ell + \tau_s$.

In [ ]:
def Gamma_trapezoid(t_arr, ell, tau_s, alpha_max=1.0):
    """Squared trapezoidal envelope (for comparison)."""
    abs_t = np.abs(t_arr)
    L = ell / 2
    S = L + tau_s
    result = np.zeros_like(abs_t)
    plateau = abs_t <= L
    ramp = (abs_t > L) & (abs_t <= S)
    result[plateau] = alpha_max**2
    result[ramp] = alpha_max**2 * ((S - abs_t[ramp]) / tau_s)**2
    return result


# Match parameters: ell=tau_plat, tau_s=tau_em=tau_dec for fair comparison
ell_trap = tau_plat
tau_s_trap = tau_em  # = tau_dec

Gamma_trap = Gamma_trapezoid(t_fine, ell_trap, tau_s_trap, alpha_max)

# R_Gamma for trapezoidal
tau_trap_fft, R_trap_fft = R_Gamma_fft(t_fine, Gamma_trap)
R_trap_norm = R_trap_fft / R_trap_fft[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Envelopes
axes[0].plot(t_fine, Gamma_fine / alpha_max**2, 'C0', lw=2, label='Sigmoid')
axes[0].plot(t_fine, Gamma_trap / alpha_max**2, 'C1', lw=2, ls='--', label='Trapezoidal')
axes[0].set_xlabel('Time [days]')
axes[0].set_ylabel(r'$\Gamma(t) / \alpha_{\max}^2$')
axes[0].set_title(r'Normalized Squared Envelopes $\Gamma(t)$')
axes[0].legend()
axes[0].set_xlim(-10, tau_plat + 10)

# Autocorrelations
axes[1].plot(tau_fft[mask], R_fft_norm[mask], 'C0', lw=2, label='Sigmoid')
mask_trap = tau_trap_fft < tau_plot_max
axes[1].plot(tau_trap_fft[mask_trap], R_trap_norm[mask_trap], 'C1', lw=2, ls='--',
             label='Trapezoidal')
axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel(r'$R_\Gamma(\tau) / R_\Gamma(0)$')
axes[1].set_title(r'Autocorrelation $R_\Gamma(\tau)$ comparison')
axes[1].legend()
axes[1].set_xlim(0, tau_plot_max)

plt.tight_layout()
plt.show()

In [ ]:
# Derivative comparison: show the kinks in the trapezoidal case
dtau = tau_fft[1] - tau_fft[0]

dR_sigmoid = np.gradient(R_fft_norm, dtau)
dR_trap = np.gradient(R_trap_norm, dtau)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(tau_fft[mask], dR_sigmoid[mask], 'C0', lw=2, label='Sigmoid')
axes[0].plot(tau_trap_fft[mask_trap], dR_trap[mask_trap], 'C1', lw=2, ls='--',
             label='Trapezoidal')
axes[0].set_xlabel(r'$\tau$ [days]')
axes[0].set_ylabel(r"$R'_\Gamma(\tau)$")
axes[0].set_title(r'First derivative of $R_\Gamma(\tau)$')
axes[0].legend()
axes[0].set_xlim(0, tau_plot_max)

# Second derivative
d2R_sigmoid = np.gradient(dR_sigmoid, dtau)
d2R_trap = np.gradient(dR_trap, dtau)

axes[1].plot(tau_fft[mask], d2R_sigmoid[mask], 'C0', lw=2, label='Sigmoid')
axes[1].plot(tau_trap_fft[mask_trap], d2R_trap[mask_trap], 'C1', lw=2, ls='--',
             label='Trapezoidal')
axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel(r"$R''_\Gamma(\tau)$")
axes[1].set_title(r'Second derivative of $R_\Gamma(\tau)$')
axes[1].legend()
axes[1].set_xlim(0, tau_plot_max)

# Mark kink locations for trapezoidal
for ax in axes:
    for kink_tau, lbl in [(tau_s_trap, r'$\tau_s$'), (ell_trap, r'$\ell$'),
                           (ell_trap + tau_s_trap, r'$\ell+\tau_s$')]:
        ax.axvline(kink_tau, color='C1', ls=':', lw=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

print("The trapezoidal R_Gamma has visible kinks in its derivatives at")
print(f"  tau_s = {tau_s_trap}, ell = {ell_trap}, ell+tau_s = {ell_trap + tau_s_trap}")
print("The sigmoid R_Gamma is smooth (C-infinity) everywhere.")

## 6. Assembling the Full Single-Spot Kernel

From Eq. (`kernel_single_spot`):
$$k(\tau) = \alpha_{\max}^4\, R_\Gamma(\tau) \left[ |c_0|^2 + 2\sum_{n=1}^{N} |c_n|^2\, \cos(n\omega_0\tau) \right]$$

where $c_n$ are the Fourier coefficients of the visibility function $\Pi(t) = \max\{\cos\beta(t),\,0\}$,
and $\omega_0 = 2\pi/P_{\rm eq}$ is the rotation frequency.

The **only** difference from the trapezoidal kernel is that $R_\Gamma(\tau)$ is now computed
numerically from the sigmoid envelope, rather than being a closed-form piecewise polynomial.

In [ ]:
def visibility_fourier_coefficients(inc, phi, n_harmonics=2):
    """Compute Fourier coefficients c_n of Pi(t) = max{cos(beta(t)), 0}.
    
    From Eq. (cn_general) in the paper.
    
    Parameters
    ----------
    inc : float
        Stellar inclination [rad].
    phi : float
        Spot latitude [rad].
    n_harmonics : int
        Number of harmonics to compute.
    
    Returns
    -------
    c : array of shape (n_harmonics + 1,)
        Fourier coefficients c_0, c_1, ..., c_N.
    """
    a0 = np.cos(inc) * np.sin(phi)
    a1 = np.sin(inc) * np.cos(phi)
    
    if np.abs(a0) >= a1:  # spot never visible or always visible
        if a0 > 0:  # always visible
            theta_vis = np.pi
        else:  # never visible
            return np.zeros(n_harmonics + 1)
    else:
        theta_vis = np.arccos(-a0 / a1)
    
    c = np.zeros(n_harmonics + 1)
    
    # c_0
    c[0] = (a0 * theta_vis + a1 * np.sin(theta_vis)) / np.pi
    
    # c_1
    c[1] = (a0 * np.sin(theta_vis) +
            a1 * (theta_vis + np.sin(theta_vis) * np.cos(theta_vis)) / 2) / np.pi
    
    # c_n for n >= 2
    for n in range(2, n_harmonics + 1):
        term1 = a0 * np.sin(n * theta_vis) / n
        term2 = a1 / 2 * (
            np.sin((n - 1) * theta_vis) / (n - 1) +
            np.sin((n + 1) * theta_vis) / (n + 1)
        )
        c[n] = (term1 + term2) / np.pi
    
    return c


def kernel_single_spot(tau_arr, R_Gamma_arr, cn, omega0, alpha_max):
    """Single-spot kernel (Eq. kernel_single_spot).
    
    k(tau) = alpha_max^4 * R_Gamma(tau) * [|c_0|^2 + 2*sum |c_n|^2 cos(n*omega0*tau)]
    """
    cosine_sum = cn[0]**2 + 2 * sum(
        cn[n]**2 * np.cos(n * omega0 * tau_arr)
        for n in range(1, len(cn))
    )
    return alpha_max**4 * R_Gamma_arr * cosine_sum

In [ ]:
# Kernel parameters
P_eq = 5.0       # rotation period [days]
inc = np.pi / 2  # edge-on
phi = np.pi / 4  # 45 deg latitude
omega0 = 2 * np.pi / P_eq
n_harmonics = 2

cn = visibility_fourier_coefficients(inc, phi, n_harmonics)
print(f"Fourier coefficients (I={np.degrees(inc):.0f} deg, Phi={np.degrees(phi):.0f} deg):")
for n, c in enumerate(cn):
    print(f"  c_{n} = {c:.6f}")

# Compute R_Gamma on the tau grid for both envelopes
tau_kernel = np.linspace(0, 40, 2000)

# Sigmoid R_Gamma (via convolution)
R_sig = R_Gamma_convolution(t_fine, Gamma_fine, tau_kernel)
R_sig_norm = R_sig / R_sig[0]

# Trapezoidal R_Gamma (via convolution)
R_trap = R_Gamma_convolution(t_fine, Gamma_trap, tau_kernel)
R_trap_norm = R_trap / R_trap[0]

# Build kernels
k_sigmoid = kernel_single_spot(tau_kernel, R_sig_norm, cn, omega0, alpha_max)
k_trapezoid = kernel_single_spot(tau_kernel, R_trap_norm, cn, omega0, alpha_max)

# Normalize for comparison
k_sigmoid_norm = k_sigmoid / k_sigmoid[0]
k_trapezoid_norm = k_trapezoid / k_trapezoid[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(tau_kernel, k_sigmoid_norm, 'C0', lw=2, label='Sigmoid')
axes[0].plot(tau_kernel, k_trapezoid_norm, 'C1', lw=2, ls='--', label='Trapezoidal')
axes[0].set_xlabel(r'$\tau$ [days]')
axes[0].set_ylabel(r'$k(\tau) / k(0)$')
axes[0].set_title(f'Normalized kernel ($P_{{eq}}$={P_eq}d, $I$={np.degrees(inc):.0f}$^\circ$, $\Phi$={np.degrees(phi):.0f}$^\circ$)')
axes[0].legend()
axes[0].axhline(0, color='gray', ls='--', lw=0.5)

# Zoom in on first few periods
axes[1].plot(tau_kernel, k_sigmoid_norm, 'C0', lw=2, label='Sigmoid')
axes[1].plot(tau_kernel, k_trapezoid_norm, 'C1', lw=2, ls='--', label='Trapezoidal')
axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel(r'$k(\tau) / k(0)$')
axes[1].set_title('Zoom: first 3 periods')
axes[1].set_xlim(0, 3 * P_eq)
axes[1].legend()
axes[1].axhline(0, color='gray', ls='--', lw=0.5)

plt.tight_layout()
plt.show()

### Check: kernel ingredients

(i) The visibility coefficients must be the Fourier coefficients
$c_n = \frac{1}{2\pi}\int_{-\pi}^{\pi}\max\{a_0 + a_1\cos\theta,\,0\}\cos(n\theta)\,d\theta$, with
$a_0 = \cos I\sin\Phi$, $a_1 = \sin I\cos\Phi$ (`eq:cn_general`). Checked by quadrature for this geometry and for
partially, always, and never visible spots.

(ii) Each $R_\Gamma$ used to build a kernel must be the autocorrelation of its own $\Gamma$. Compare against an
independent Wiener–Khinchin estimate of the same sampled $\Gamma$, zero-padded to $2N$ so the FFT gives the linear
(non-circular) autocorrelation, over the full lag grid. The tolerance is $10^{-5}R_\Gamma(0)$. The sigmoid agrees
to $\sim10^{-6}$; that residual arises because `alpha_sigmoid` renormalizes each shifted copy by its maximum on
the shifted grid, which no longer contains the peak for $\tau \gtrsim 31.5$ d.

In [ ]:
theta_q = np.linspace(-np.pi, np.pi, 200001)
for inc_c, phi_c in [(inc, phi), (np.radians(60), np.radians(30)), (np.radians(60), np.radians(-50)),
                     (np.radians(30), np.radians(70)), (np.radians(20), np.radians(-80))]:
    a0_c, a1_c = np.cos(inc_c) * np.sin(phi_c), np.sin(inc_c) * np.cos(phi_c)
    Pi_q = np.maximum(a0_c + a1_c * np.cos(theta_q), 0.0)
    cn_quad = np.array([np.trapezoid(Pi_q * np.cos(n * theta_q), theta_q) / (2 * np.pi)
                        for n in range(n_harmonics + 1)])
    cn_c = visibility_fourier_coefficients(inc_c, phi_c, n_harmonics)
    # observed agreement ~1e-11 (quadrature error at the visibility kinks)
    assert np.allclose(cn_c, cn_quad, rtol=0, atol=1e-8), (np.degrees([inc_c, phi_c]), cn_c, cn_quad)
print("Verified: c_n match direct quadrature of the visibility function.")


def R_Gamma_wk_linear(t_arr, Gamma_arr, tau_eval):
    # Linear autocorrelation of the sampled Gamma via zero-padded FFT, interpolated at tau_eval >= 0
    dt = t_arr[1] - t_arr[0]
    N = len(t_arr)
    F = np.fft.rfft(Gamma_arr, n=2 * N)
    R = np.fft.irfft(np.abs(F)**2, n=2 * N)[:N] * dt
    return np.interp(tau_eval, np.arange(N) * dt, R)


R_sig_wk = R_Gamma_wk_linear(t_fine, Gamma_fine, tau_kernel)
R_trap_wk = R_Gamma_wk_linear(t_fine, Gamma_trap, tau_kernel)
err_sig = np.max(np.abs(R_sig - R_sig_wk)) / R_sig_wk[0]
err_trap = np.max(np.abs(R_trap - R_trap_wk)) / R_trap_wk[0]
print(f"max |R - R_WK| / R(0): sigmoid {err_sig:.2e}, trapezoid {err_trap:.2e}")

assert err_sig < 1e-5, f"sigmoid R_Gamma used in the kernel is not the autocorrelation of Gamma ({err_sig:.2e})"
assert err_trap < 1e-5, (
    f"trapezoidal R_Gamma used in the kernel is not the autocorrelation of Gamma_trap ({err_trap:.2e} of R(0); "
    f"R_trap peaks at tau = {tau_kernel[np.argmax(R_trap)]:.2f} d instead of 0). R_Gamma_convolution ignores its "
    "Gamma_arr argument for the shifted copy and always uses Gamma_sigmoid(t + tau, tau_em, tau_plat, tau_dec, "
    "alpha_max), so this is the trapezoid-sigmoid cross-correlation.")

## 7. Parameter Sensitivity

Explore how the sigmoid kernel varies with the emergence/decay timescales and plateau duration.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Vary tau_em
for te in [1.0, 3.0, 8.0]:
    t_tmp = np.linspace(-8 * te, tau_plat + 8 * max(te, tau_dec), 2**14)
    G_tmp = Gamma_sigmoid(t_tmp, te, tau_plat, tau_dec, alpha_max)
    R_tmp = R_Gamma_convolution(t_tmp, G_tmp, tau_kernel)
    k_tmp = kernel_single_spot(tau_kernel, R_tmp / R_tmp[0], cn, omega0, alpha_max)
    axes[0].plot(tau_kernel, k_tmp / k_tmp[0], lw=2, label=rf'$\tau_{{\rm em}}={te}$')
axes[0].set_title(r'Vary $\tau_{\rm em}$')

# Vary tau_plat
for tm in [5.0, 15.0, 30.0]:
    t_tmp = np.linspace(-8 * tau_em, tm + 8 * tau_dec, 2**14)
    G_tmp = Gamma_sigmoid(t_tmp, tau_em, tm, tau_dec, alpha_max)
    R_tmp = R_Gamma_convolution(t_tmp, G_tmp, tau_kernel)
    k_tmp = kernel_single_spot(tau_kernel, R_tmp / R_tmp[0], cn, omega0, alpha_max)
    axes[1].plot(tau_kernel, k_tmp / k_tmp[0], lw=2, label=rf'$\tau_{{\rm plat}}={tm}$')
axes[1].set_title(r'Vary $\tau_{\rm plat}$ (plateau duration)')

# Vary tau_dec
for td in [1.0, 3.0, 8.0]:
    t_tmp = np.linspace(-8 * tau_em, tau_plat + 8 * td, 2**14)
    G_tmp = Gamma_sigmoid(t_tmp, tau_em, tau_plat, td, alpha_max)
    R_tmp = R_Gamma_convolution(t_tmp, G_tmp, tau_kernel)
    k_tmp = kernel_single_spot(tau_kernel, R_tmp / R_tmp[0], cn, omega0, alpha_max)
    axes[2].plot(tau_kernel, k_tmp / k_tmp[0], lw=2, label=rf'$\tau_{{\rm dec}}={td}$')
axes[2].set_title(r'Vary $\tau_{\rm dec}$')

for ax in axes:
    ax.set_xlabel(r'$\tau$ [days]')
    ax.set_ylabel(r'$k(\tau)/k(0)$')
    ax.legend(fontsize=10)
    ax.axhline(0, color='gray', ls='--', lw=0.5)

plt.tight_layout()
plt.show()

## 8. Asymmetric Emergence/Decay

Unlike the trapezoidal envelope used elsewhere in the paper (which assumes $\tau_{\rm em} = \tau_{\rm dec} = \tau_{\rm spot}$),
the sigmoid naturally accommodates **asymmetric** emergence and decay timescales.
This is physically motivated: solar observations show that spots emerge faster than they decay.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cases = [
    (1.0, 5.0, r'$\tau_{\rm em}=1, \tau_{\rm dec}=5$ (fast rise, slow decay)'),
    (3.0, 3.0, r'$\tau_{\rm em}=3, \tau_{\rm dec}=3$ (symmetric)'),
    (5.0, 1.0, r'$\tau_{\rm em}=5, \tau_{\rm dec}=1$ (slow rise, fast decay)'),
]

for te, td, label in cases:
    t_tmp = np.linspace(-8 * te, tau_plat + 8 * td, 2**14)
    alpha_tmp = alpha_sigmoid(t_tmp, te, tau_plat, td, alpha_max)
    G_tmp = alpha_tmp**2
    axes[0].plot(t_tmp, alpha_tmp / alpha_max, lw=2, label=label)
    
    R_tmp = R_Gamma_convolution(t_tmp, G_tmp, tau_kernel)
    k_tmp = kernel_single_spot(tau_kernel, R_tmp / R_tmp[0], cn, omega0, alpha_max)
    axes[1].plot(tau_kernel, k_tmp / k_tmp[0], lw=2, label=label)

axes[0].set_xlabel('Time [days]')
axes[0].set_ylabel(r'$\alpha(t) / \alpha_{\max}$')
axes[0].set_title('Asymmetric sigmoid envelopes')
axes[0].legend(fontsize=9)
axes[0].set_xlim(-15, tau_plat + 15)

axes[1].set_xlabel(r'$\tau$ [days]')
axes[1].set_ylabel(r'$k(\tau)/k(0)$')
axes[1].set_title('Resulting kernels')
axes[1].legend(fontsize=9)
axes[1].axhline(0, color='gray', ls='--', lw=0.5)

plt.tight_layout()
plt.show()

print("Note: the kernel k(tau) is always symmetric in tau (as required for a")
print("valid covariance function), even when the envelope is asymmetric.")
print("This is because R_Gamma(tau) = R_Gamma(-tau) by construction (autocorrelation).")

### Check: $R_\Gamma$ in Sections 7–8

Each kernel in Sections 7–8 uses `R_Gamma_convolution` for a different $(\tau_{\rm em}, \tau_{\rm plat}, \tau_{\rm dec})$
on its own time grid. Recompute every case (on every 20th lag) and require agreement with the zero-padded
Wiener–Khinchin autocorrelation of the same $\Gamma$, as above. For the asymmetric envelopes, also verify the claim
printed above: the convolution evaluated at $-\tau$ must give the same $R_\Gamma(\tau)$.

In [ ]:
tau_chk = tau_kernel[::20]
sweep = []
for te in [1.0, 3.0, 8.0]:        # Section 7, vary tau_em
    sweep.append((te, tau_plat, tau_dec, np.linspace(-8 * te, tau_plat + 8 * max(te, tau_dec), 2**14)))
for tm in [5.0, 15.0, 30.0]:      # Section 7, vary tau_plat
    sweep.append((tau_em, tm, tau_dec, np.linspace(-8 * tau_em, tm + 8 * tau_dec, 2**14)))
for td in [1.0, 3.0, 8.0]:        # Section 7, vary tau_dec
    sweep.append((tau_em, tau_plat, td, np.linspace(-8 * tau_em, tau_plat + 8 * td, 2**14)))
for te, td, _ in cases:           # Section 8, asymmetric cases
    sweep.append((te, tau_plat, td, np.linspace(-8 * te, tau_plat + 8 * td, 2**14)))

failures = []
for te_c, tm_c, td_c, t_c in sweep:
    G_c = Gamma_sigmoid(t_c, te_c, tm_c, td_c, alpha_max)
    R_ref = R_Gamma_wk_linear(t_c, G_c, tau_chk)
    err_pos = np.max(np.abs(R_Gamma_convolution(t_c, G_c, tau_chk) - R_ref)) / R_ref[0]
    err_neg = 0.0
    if te_c != td_c:
        err_neg = np.max(np.abs(R_Gamma_convolution(t_c, G_c, -tau_chk) - R_ref)) / R_ref[0]
    print(f"tau_em={te_c}, tau_plat={tm_c}, tau_dec={td_c}: err(+tau) = {err_pos:.2e}, err(-tau) = {err_neg:.2e}")
    if err_pos >= 1e-5 or err_neg >= 1e-5:
        failures.append((te_c, tm_c, td_c, err_pos, err_neg))

assert not failures, (
    f"R_Gamma_convolution is not the autocorrelation of its Gamma for (tau_em, tau_plat, tau_dec, err+, err-) = "
    f"{failures}. It builds the shifted copy from Gamma_sigmoid(t + tau, tau_em, tau_plat, tau_dec, alpha_max) with "
    "the global example parameters, ignoring Gamma_arr.")

## Summary

The sigmoid envelope (Eq. 11) yields a kernel with the **same factorized structure** as the
trapezoidal case:

$$k(\tau) = \alpha_{\max}^4\, R_\Gamma(\tau) \left[ |c_0|^2 + 2\sum_{n=1}^{N} |c_n|^2\, \cos(n\omega_0\tau) \right]$$

The visibility coefficients $c_n$ are unchanged (they depend only on $(I, \Phi)$, not the
envelope shape). The only difference is in $R_\Gamma(\tau)$:

| Property | Trapezoidal | Sigmoid |
|---|---|---|
| $\Gamma(t) = \alpha^2(t)$ | Piecewise polynomial (parabolic ramps) | Product of squared logistic functions |
| $\hat{\Gamma}(\omega)$ | Closed-form (Eq. `Gamma_hat`) | Numerical (FFT) |
| $R_\Gamma(\tau)$ | Piecewise degree-5 polynomial | Numerical (smooth, $C^\infty$) |
| Smoothness | $C^1$ at $\tau=0$; kinks at $\tau_s$, $\ell$, $\ell+\tau_s$ | $C^\infty$ everywhere |
| Asymmetric rise/decay | Not supported ($\tau_{\rm em} = \tau_{\rm dec}$) | Naturally supported |
| Computational cost | $O(1)$ evaluation | $O(N_{\rm grid} \log N_{\rm grid})$ via FFT, then interpolation |

For GP inference, $R_\Gamma(\tau)$ can be precomputed on a fine grid and interpolated,
making the sigmoid kernel only marginally more expensive than the trapezoidal kernel in practice.